In [0]:
%sql
--create catalog airflow_gold

In [0]:
%sql
--create schema airflow_gold.gold

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format('delta')\
    .load("s3://awsbucketmoni2026/Silver/obt")
display(df)

booking_id,passenger_id,flight_id,airport_id,amount,booking_date,airport_name,city,country,name,gender,nationality,Processed_Date
B00001,P0048,F0014,A048,850.72,2025-05-29,Sarahmouth International Airport,Edwardmouth,Malaysia,Jonathan Martinez,Male,Mayotte,2026-06-20T11:40:51.696Z
B00002,P0011,F0052,A003,376.63,2025-06-09,Brownland International Airport,Samuelville,Costa Rica,Michael Anderson MD,Female,Czech Republic,2026-06-20T11:40:51.696Z
B00003,P0079,F0023,A012,534.02,2025-06-03,East Nancy International Airport,Crosston,Romania,Tracy Russo,Male,Iceland,2026-06-20T11:40:51.696Z
B00004,P0068,F0001,A039,1333.7,2025-06-16,Reneeborough International Airport,East Amanda,Germany,Timothy Goodwin,Female,Kuwait,2026-06-20T11:40:51.696Z
B00005,P0189,F0019,A008,1334.96,2025-06-17,Port Craig International Airport,New Lisa,French Southern Territories,Joel Gomez,Female,Bulgaria,2026-06-20T11:40:51.696Z
B00006,P0070,F0003,A004,296.13,2025-05-18,Meghanton International Airport,Andrewsmouth,Macedonia,Anthony Ingram,Male,Papua New Guinea,2026-06-20T11:40:51.696Z
B00007,P0194,F0068,A016,460.14,2025-04-05,Davidhaven International Airport,South Michael,Holy See (Vatican City State),Tommy Baker,Male,Venezuela,2026-06-20T11:40:51.696Z
B00008,P0181,F0048,A017,1402.02,2025-06-04,West Melissaborough International Airport,North Robertburgh,Switzerland,Ryan Jenkins,Female,Lao People's Democratic Republic,2026-06-20T11:40:51.696Z
B00009,P0096,F0081,A016,1444.51,2025-05-16,Davidhaven International Airport,South Michael,Holy See (Vatican City State),Erika Hall,Male,Kenya,2026-06-20T11:40:51.696Z
B00010,P0131,F0089,A034,292.39,2025-05-16,West Matthew International Airport,Wigginsmouth,Czech Republic,Dr. Kyle Salazar,Male,Brazil,2026-06-20T11:40:51.696Z


### DIM_PASSENGER - SCD Type 1

In [0]:
# Extract unique passengers from OBT
df_passenger = df.select(
    col("passenger_id"),
    col("name"),
    col("gender"),
    col("nationality")
).distinct()

# Add surrogate key using row_number
window_spec = Window.orderBy("passenger_id")
df_passenger = df_passenger.withColumn("passenger_sk", row_number().over(window_spec))

# Reorder columns: surrogate key first
df_passenger = df_passenger.select(
    col("passenger_sk"),
    col("passenger_id"),
    col("name"),
    col("gender"),
    col("nationality")
)

# Write to Gold layer (full overwrite - Type 1)
df_passenger.write.format('delta') \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("airflow_gold.gold.dim_passenger")

print(f"Created dim_passenger with {df_passenger.count()} records")
display(df_passenger)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Created dim_passenger with 200 records


passenger_sk,passenger_id,name,gender,nationality
1,P0001,Kevin Ferguson,Male,Reunion
2,P0002,Kathleen Martinez DVM,Female,Burkina Faso
3,P0003,Cynthia Frazier,Male,Marshall Islands
4,P0004,Ryan Ramsey,Male,Niger
5,P0005,Mike Kim,Male,Taiwan
6,P0006,Diana Adams,Male,Mayotte
7,P0007,Sharon Moon,Male,Madagascar
8,P0008,Cheryl Glenn,Male,Maldives
9,P0009,Allen Lowery,Male,Rwanda
10,P0010,Maria Medina,Male,Denmark


### DIM_AIRPORT - SCD Type 1


In [0]:
# Extract unique airports from OBT
df_airport = df.select(
    col("airport_id"),
    col("airport_name"),
    col("city"),
    col("country")
).distinct()

# Add surrogate key using row_number
window_spec = Window.orderBy("airport_name")
df_airport = df_airport.withColumn("airport_sk", row_number().over(window_spec))

# Reorder columns: surrogate key first
df_airport = df_airport.select(
    col("airport_sk"),
    col("airport_id"),
    col("airport_name"),
    col("city"),
    col("country")
)

# Write to Gold layer (full overwrite - Type 1)
df_airport.write.format('delta') \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("airflow_gold.gold.dim_airport")

print(f"Created dim_airport with {df_airport.count()} records")
display(df_airport)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Created dim_airport with 50 records


airport_sk,airport_id,airport_name,city,country
1,A011,Alisonville International Airport,West Annaburgh,Mexico
2,A022,Blankenshipport International Airport,Sherryville,Bhutan
3,A014,Brittanyshire International Airport,Kylechester,Mauritius
4,A003,Brownland International Airport,Samuelville,Costa Rica
5,A035,Chavezchester International Airport,Port Meganhaven,Niger
6,A044,Cordovaview International Airport,Shannonport,Gabon
7,A016,Davidhaven International Airport,South Michael,Holy See (Vatican City State)
8,A005,East Aaron International Airport,Davishaven,Monaco
9,A019,East Gregtown International Airport,Downsborough,Kazakhstan
10,A002,East Kristin International Airport,North Michaelview,Bosnia and Herzegovina


### DIM_DATE - Date Dimension


In [0]:
# Extract unique dates from OBT
df_dates = df.select(col("booking_date")).distinct()

# Generate date attributes
df_date = df_dates.select(
    date_format(col("booking_date"), "yyyyMMdd").cast("int").alias("date_key"),
    col("booking_date"),
    year(col("booking_date")).alias("year"),
    quarter(col("booking_date")).alias("quarter"),
    month(col("booking_date")).alias("month"),
    date_format(col("booking_date"), "MMMM").alias("month_name"),
    dayofmonth(col("booking_date")).alias("day_of_month"),
    dayofweek(col("booking_date")).alias("day_of_week"),
    date_format(col("booking_date"), "EEEE").alias("day_name"),
    when(dayofweek(col("booking_date")).isin([1, 7]), True).otherwise(False).alias("is_weekend"),
    weekofyear(col("booking_date")).alias("week_of_year")
)

# Write to Gold layer (full overwrite)
df_date.write.format('delta') \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("airflow_gold.gold.dim_date")

print(f"Created dim_date with {df_date.count()} records")
display(df_date)

Created dim_date with 90 records


date_key,booking_date,year,quarter,month,month_name,day_of_month,day_of_week,day_name,is_weekend,week_of_year
20250324,2025-03-24,2025,1,3,March,24,2,Monday,false,13
20250411,2025-04-11,2025,2,4,April,11,6,Friday,false,15
20250406,2025-04-06,2025,2,4,April,6,1,Sunday,true,14
20250404,2025-04-04,2025,2,4,April,4,6,Friday,false,14
20250523,2025-05-23,2025,2,5,May,23,6,Friday,false,21
20250519,2025-05-19,2025,2,5,May,19,2,Monday,false,21
20250509,2025-05-09,2025,2,5,May,9,6,Friday,false,19
20250427,2025-04-27,2025,2,4,April,27,1,Sunday,true,17
20250426,2025-04-26,2025,2,4,April,26,7,Saturday,true,17
20250401,2025-04-01,2025,2,4,April,1,3,Tuesday,false,14


## FACT_BOOKING - Fact Table

Join OBT with dimensions to get surrogate keys

In [0]:
# Read dimension tables to get surrogate keys
df_dim_passenger = spark.read.table("airflow_gold.gold.dim_passenger") \
    .select(col("passenger_sk"), col("passenger_id"))

df_dim_airport = spark.read.table("airflow_gold.gold.dim_airport") \
    .select(col("airport_sk"), col("airport_id"))

df_dim_date = spark.read.table("airflow_gold.gold.dim_date") \
    .select(col("date_key"), col("booking_date"))

# Join OBT with dimensions to create fact table
df_fact = df.select(
    col("booking_id"),
    col("passenger_id"),
    col("airport_id"),
    col("flight_id"),
    col("booking_date"),
    col("amount")
)

# Join with passenger dimension
df_fact = df_fact.join(df_dim_passenger, "passenger_id", "left") \
    .drop("passenger_id")

# Join with airport dimension
df_fact = df_fact.join(df_dim_airport, "airport_id", "left") \
    .drop("airport_id")

# Join with date dimension
df_fact = df_fact.join(df_dim_date, "booking_date", "left")

# Select final fact table columns
df_fact_booking = df_fact.select(
    col("booking_id"),
    col("passenger_sk"),
    col("airport_sk"),
    col("flight_id"),
    col("date_key"),
    col("amount"),
    col("booking_date")  # partition column
)

# Write to Gold layer (full overwrite)
df_fact_booking.write.format('delta') \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("booking_date") \
    .saveAsTable("airflow_gold.gold.fact_booking")

print(f"Created fact_booking with {df_fact_booking.count()} records")
display(df_fact_booking)

Created fact_booking with 1000 records


booking_id,passenger_sk,airport_sk,flight_id,date_key,amount,booking_date
B00001,48,39,F0014,20250529,850.72,2025-05-29
B00002,11,4,F0052,20250609,376.63,2025-06-09
B00003,79,11,F0023,20250603,534.02,2025-06-03
B00004,68,38,F0001,20250616,1333.7,2025-06-16
B00005,189,36,F0019,20250617,1334.96,2025-06-17
B00006,70,25,F0003,20250518,296.13,2025-05-18
B00007,194,7,F0068,20250405,460.14,2025-04-05
B00008,181,49,F0048,20250604,1402.02,2025-06-04
B00009,96,7,F0081,20250516,1444.51,2025-05-16
B00010,131,47,F0089,20250516,292.39,2025-05-16
